# Week 4 Exercise — Python Toolbox

Build a tool that uses an LLM to work with Python code:

1. **Docstring & comments:** Paste a function → get it back with a PEP-257 docstring and helpful inline comments.
2. **Unit tests:** Paste a function → get pytest (or unittest) tests covering typical, edge, and error cases.

Uses **OpenAI only** (no extra API keys). Gradio UI with two tabs and optional streaming.

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

MODEL = "gpt-4.1-mini"
client = OpenAI() if api_key else None

In [ ]:
SYSTEM_PROMPT_DOC = """You are an expert Python developer and code reviewer.
Your job is to read the user's provided function, and return:
1. A concise, PEP-257-compliant docstring summarizing what the function does, clarifying types, parameters, return values, and side effects.
2. Helpful inline comments that improve both readability and maintainability, without restating what the code obviously does.

Only output the function, not explanations or additional text.
Do not modify variable names or refactor the function logic.
Your response should improve the code's clarity and documentation, making it easier for others to understand and maintain.
Don't be extremely verbose. Your answer should be at a senior level of expertise.
"""

SYSTEM_PROMPT_TESTS = """You are a seasoned Python developer and testing expert.
Your task is to read the user's provided function, and generate:
1. A concise set of meaningful unit tests that thoroughly validate the function's correctness, including typical, edge, and error cases.
2. The tests should be written for pytest (or unittest if pytest is not appropriate), use clear, descriptive names, and avoid unnecessary complexity.
3. If dependencies or mocking are needed, include minimal necessary setup code (but avoid over-mocking).

Only output the relevant test code, not explanations or extra text.
Do not change the original function; focus solely on comprehensive, maintainable test coverage that other developers can easily understand and extend.
"""

In [ ]:
def _strip_code_block(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    for marker in ("```python", "```"):
        if t.startswith(marker):
            t = t[len(marker):].lstrip()
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t

def generate_documentation(code: str):
    """Stream docstring + comments for the given Python function."""
    if not code or not code.strip():
        yield "Paste a Python function above."
        return
    if not client:
        yield "OpenAI client not initialized. Set OPENAI_API_KEY."
        return
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_DOC},
                {"role": "user", "content": code},
            ],
            stream=True,
        )
        output = ""
        for chunk in stream:
            part = chunk.choices[0].delta.content or ""
            output += part
            yield _strip_code_block(output)
    except Exception as e:
        yield f"Error: {e}"

def generate_tests(code: str):
    """Stream unit tests for the given Python function."""
    if not code or not code.strip():
        yield "Paste a Python function above."
        return
    if not client:
        yield "OpenAI client not initialized. Set OPENAI_API_KEY."
        return
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_TESTS},
                {"role": "user", "content": code},
            ],
            stream=True,
        )
        output = ""
        for chunk in stream:
            part = chunk.choices[0].delta.content or ""
            output += part
            yield _strip_code_block(output)
    except Exception as e:
        yield f"Error: {e}"

In [ ]:
with gr.Blocks(title="Week 4 — Python Toolbox") as demo:
    gr.Markdown("# Python Toolbox")
    gr.Markdown("Generate **docstrings + comments** or **pytest unit tests** for your Python function. Paste code below and click Generate.")

    with gr.Tab("Docstring & comments"):
        gr.Markdown("### Docstring & comment generator")
        doc_in = gr.Code(label="Python function", lines=18, language="python")
        doc_btn = gr.Button("Generate docstring & comments")
        doc_out = gr.Code(label="Function with docstring and comments", language="python")
        doc_btn.click(fn=generate_documentation, inputs=[doc_in], outputs=doc_out)

    with gr.Tab("Unit tests"):
        gr.Markdown("### Unit test generator")
        test_in = gr.Code(label="Python function", lines=18, language="python")
        test_btn = gr.Button("Generate unit tests")
        test_out = gr.Code(label="Generated tests (pytest)", language="python")
        test_btn.click(fn=generate_tests, inputs=[test_in], outputs=test_out)

    gr.Examples(
        examples=[["def add(a: int, b: int) -> int:\n    return a + b"], ["def divide(a: float, b: float) -> float:\n    return a / b"]],
        inputs=doc_in,
        label="Examples",
    )

demo.launch()